# Portofolio Data Science - Pertemuan 12
- **Nama Lengkap**: Muhammad Ikctiar Saputra
- **NIM**: 250401020169
- **Kelas**: IF401
- **Program Studi**: PJJ Informatika

---

## Langkah 1: Simulasi Transaksi dan Visualisasi Frekuensi Produk

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Menentukan seed acak untuk simulasi transaksi
np.random.seed(42)

# Daftar produk yang dijual
daftar_produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Kopi', 'Gula', 'Teh', 'Keju', 'Mentega', 'Madu']

# Menghasilkan 50 transaksi acak (tiap transaksi membeli 2 - 5 produk unik)
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    item_terpilih = list(np.random.choice(daftar_produk, n_item, replace=False))
    transaksi.append(item_terpilih)

# Menyisipkan pola tersembunyi (Assosiasi): Jika membeli Kopi, maka cenderung membeli Gula
for i in range(25):
    if 'Kopi' in transaksi[i] and 'Gula' not in transaksi[i]:
        transaksi[i].append('Gula')

print("Jumlah total transaksi:", len(transaksi))
print("5 Transaksi Pertama:")
print(transaksi[:5])

# Menghitung frekuensi kemunculan produk
frekuensi = Counter()
for t in transaksi:
    frekuensi.update(t)

df_frekuensi = pd.DataFrame(frekuensi.items(), columns=['Produk', 'Jumlah']).sort_values('Jumlah', ascending=False)

# Visualisasi frekuensi produk
plt.figure(figsize=(10, 5))
plt.bar(df_frekuensi['Produk'], df_frekuensi['Jumlah'], color='darkgreen')
plt.title('Frekuensi Penjualan Produk dalam Transaksi')
plt.xlabel('Produk')
plt.ylabel('Jumlah Pembelian')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Langkah 2: Market Basket Analysis Menggunakan Apriori

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Transformasi transaksi ke format one-hot encoding array
encoder = TransactionEncoder()
array_encoded = encoder.fit(transaksi).transform(transaksi)
df_transaksi = pd.DataFrame(array_encoded, columns=encoder.columns_)

# Mencari frequent itemsets dengan minimum support 0.1
frequent_itemsets = apriori(df_transaksi, min_support=0.1, use_colnames=True)

# Mengekstrak aturan asosiasi dengan minimum lift 1.0
aturan_asosiasi = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
aturan_teratas = aturan_asosiasi.sort_values('lift', ascending=False).head(10)

print("10 Aturan Asosiasi Terbaik (Berdasarkan Lift Score):")
print(aturan_teratas[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

## Langkah 3: Sistem Rekomendasi Berbasis Konten (Content-Based Filtering)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Membuat katalog produk dan kategorinya
katalog_produk = pd.DataFrame({
    'produk': daftar_produk,
    'kategori': ['Bakery', 'Olesan', 'Dairy', 'Bakery', 'Minuman', 'Bahan Pokok', 'Minuman', 'Dairy', 'Dairy', 'Olesan']
})

# One-Hot Encoding kategori
fitur_kategori = pd.get_dummies(katalog_produk['kategori'])

# Menghitung matriks similarity menggunakan Cosine Similarity
matriks_sim = cosine_similarity(fitur_kategori)

# Fungsi untuk memberikan rekomendasi produk serupa
def dapatkan_rekomendasi_serupa(nama_produk, top_n=3):
    idx_produk = katalog_produk.index[katalog_produk['produk'] == nama_produk][0]
    skor_kemiripan = list(enumerate(matriks_sim[idx_produk]))
    skor_kemiripan = sorted(skor_kemiripan, key=lambda x: x[1], reverse=True)
    
    # Menyaring produk target itu sendiri
    skor_kemiripan = [s for s in skor_kemiripan if s[0] != idx_produk]
    
    rekomendasi = katalog_produk.iloc[[i for i, _ in skor_kemiripan[:top_n]]]['produk'].tolist()
    return rekomendasi

## Langkah 4: Perbandingan Rekomendasi untuk Produk Target (Contoh: Kopi)

In [ ]:
produk_target = "Kopi"

# Rekomendasi berdasarkan Association Rules (MBA)
rules_kopi = aturan_asosiasi[aturan_asosiasi['antecedents'].apply(lambda x: produk_target in x)]
rekomendasi_mba = rules_kopi.sort_values('lift', ascending=False)['consequents'].head(3).tolist()
rekomendasi_mba_list = [list(item)[0] for item in rekomendasi_mba]

# Rekomendasi berdasarkan Content-Based Filtering
rekomendasi_content = dapatkan_rekomendasi_serupa(produk_target, top_n=3)

print(f"=== REKOMENDASI UNTUK PRODUK: {produk_target} ===")
print("1. Berdasarkan Pembelian Bersamaan (Market Basket Analysis):", rekomendasi_mba_list)
print("2. Berdasarkan Kemiripan Kategori Produk (Content-Based):", rekomendasi_content)

## Kesimpulan & Pembahasan

Berdasarkan analisis keranjang belanja dan sistem rekomendasi:
1. **Market Basket Analysis**: Dengan menetapkan minimum support 0.1 dan lift 1.0, didapatkan aturan asosiasi terkuat. Pola asosiasi buatan `Kopi -> Gula` berhasil terdeteksi dengan nilai *lift* yang tinggi, yang membuktikan bahwa pelanggan yang membeli kopi memiliki kecenderungan kuat untuk membeli gula secara bersamaan.
2. **Perbedaan Pendekatan Rekomendasi**:
   - **Market Basket Analysis (MBA)** merekomendasikan produk berdasarkan perilaku transaksi nyata (produk yang sering dibeli bersama), meskipun produk tersebut berbeda kategori (misal: Kopi (Minuman) direkomendasikan dengan Gula (Bahan Pokok)).
   - **Content-Based Filtering** memberikan rekomendasi berdasarkan karakteristik atau kemiripan atribut produk (misal: Kopi direkomendasikan dengan Teh karena keduanya berada dalam kategori Minuman).
3. **Penerapan Bisnis**: Hasil MBA dapat digunakan untuk penataan produk di rak supermarket (cross-merchandising), sedangkan Content-Based sangat cocok digunakan untuk memunculkan produk substitusi/serupa di halaman detail e-commerce.